In [1]:
from utils.misc import suppress_warnings

suppress_warnings()

import pandas as pd

from model.llm import build_language_model

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
print(f"Loading {model_name}...")

# Load the model and tokenizer
model, tokenizer = build_language_model(model_name)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
ACTION_PROMPTS = {
    "clarification": lambda _: "You are part of a movie recommendation system. The user's request is too vague to provide a recommendation. In a CONCISE way, (i.e. limited to two questions or less) ask the user for more information about their preferences. Reply ONLY with the human-like request for more information. DO NOT include any other text.",
    "recommendation": lambda movie: f"You are a part of a movie recommendation system. The user has requested a movie recommendation. Recommend the movie '{movie}' to the user. Include a brief description of why the movie matches the user's request. Reply ONLY with the movie recommendation and description. DO NOT include any other text."
}

PROMPT_GENERATORS = [
    # Normal
    lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to craft a casual request that will help the system suggest the movie "{movie}" without mentioning its title, characters, or ANY plot elements. Focus on broad genres and simple preferences. For example, for the movie "The Godfather," you might say, "Looking for a classic crime drama." Reply ONLY with the human-like request for a movie. DO NOT include any other text.""",
    # Mood/Feeling
    lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to describe the mood you're in, which should lead the system to suggest the movie "{movie}" without including any specific details. Keep it conversational and focus on your feelings or current vibe. For example, for the movie "The Notebook," you could say, "I'm in the mood for something romantic and emotional." Reply ONLY with the human-like request for a movie. DO NOT include any other text.""",
    # Country, Time Period, Critical Acclaim
    lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to provide a request that reflects a niche preference while guiding the system to suggest the movie "{movie}" without mentioning any plot details. Emphasize aspects like country of origin, awards, critical acclaim, or the time period in which the movie was released (e.g., early 2000s, or use descriptors like new/old). For example, for the movie "Parasite," you could say, "Can you suggest a recent Oscar-winning foreign thriller?" Reply ONLY with the human-like request for a movie. DO NOT include any other text.""",
    # Style, Pacing, Theme
    lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to make a precise yet intriguing request that will help the system suggest the movie "{movie}" without referring to its plot or characters. Highlight its pacing, storytelling style, or themes. For example, for the movie "Pulp Fiction," you could say, "I want something edgy and non-linear with a lot of dialogue." Reply ONLY with the human-like request for a movie. DO NOT include any other text.""",
    # Specific Actor
    lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to create a request that will help the system recommend the movie "{movie}" without mentioning its title, characters, or plot. Instead, focus on requesting a movie featuring a specific actor from the movie. For example, for the movie "National Treasure," you might say, "I want to watch something with Nicolas Cage in it." Reply ONLY with the human-like request for a movie. DO NOT include any other text.""",
    # Similar Movie
    lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to create a request that will guide the system to suggest the movie "{movie}" without mentioning its title, characters, or plot. Instead, base your request on a similar movie that shares key characteristics like genre, tone, or themes. For example, for the movie "The Dark Knight," you might say, "I'm looking for something like 'Batman Begins,' with a dark and gritty superhero vibe." Reply ONLY with the human-like request for a movie. DO NOT include any other text.""",
]


In [3]:
data_path = "data/multi-turn/ml-20m/movies.csv"
output_path = "data/multi-turn/ml-20m/conversations.csv"

# Read in the data
movies = pd.read_csv(data_path, header=0, names=["item_id", "item_title", "genres"])

movies = movies[["item_id", "item_title"]]

data = pd.DataFrame(columns=["item_id", "item_title", "request"])

In [5]:
movies

,item_id,item_title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)
...,...,...
27273,131254,Kein Bund für's Leben (2007)
27274,131256,"Feuer, Eis & Dosenbier (2002)"
27275,131258,The Pirates (2014)
27276,131260,Rentun Ruusu (2001)


In [7]:
from transformers import LlamaTokenizer

In [ ]:

prompt_generator = lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to craft a casual request that will help the system suggest the movie "{movie}" without mentioning its title, characters, or ANY plot elements. Focus on broad genres and simple preferences. For example, for the movie "The Godfather," you might say, "Looking for a classic crime drama." Reply ONLY with the human-like request for a movie. DO NOT include any other text."""

prompt_generator = lambda movie: f"""You are interacting with a movie recommendation system. Your goal is to craft an extremely vague request for a movie that does not give the system enough information to give a recommendation. Reply ONLY with the human-like request for a movie. DO NOT include any other text."""

prompt = prompt_generator("Parasite")

# Form prompt
chat = [{"role": "system", "content": prompt}]

tokenizer: LlamaTokenizer = tokenizer

# Apply the chat template
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# Tokenize input
batch_input_tokens = tokenizer(prompt, padding=True, return_tensors="pt").to(model.device)

# Generate response
batch_output_tokens = model.generate(**batch_input_tokens, max_new_tokens=64, do_sample=True)

# Decode response
responses = [
    tokenizer.decode(output_tokens[len(input_tokens) :], skip_special_tokens=True).strip('"')
    for input_tokens, output_tokens in zip(batch_input_tokens["input_ids"], batch_output_tokens)
]

print(responses[0])

In [ ]:
chat = [
    {"role": "user", "content": responses[0]},
    {"role": "system", "content": "You are part of a movie recommendation system. The user's request is too vague to provide a recommendation. In a CONCISE way, (i.e. limited to two questions or less) ask the user for more information about their preferences. Reply ONLY with the human-like request for more information. DO NOT include any other text."},
]

prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# Tokenize input
batch_input_tokens = tokenizer(prompt, padding=True, return_tensors="pt").to(model.device)

# Generate response
batch_output_tokens = model.generate(**batch_input_tokens, max_new_tokens=256, do_sample=True)

# Decode response
responses = [
    tokenizer.decode(output_tokens[len(input_tokens) :], skip_special_tokens=True).strip('"')
    for input_tokens, output_tokens in zip(batch_input_tokens["input_ids"], batch_output_tokens)
]

print(responses[0])

In [ ]:
chat = [
    {"role": "user", "content": responses[0]},
    {"role": "system", "content": "Give a concise response to the user's request with a recommendation for 'Past Lives' consisting of the following parts: The recommendation for the movie 'Past Lives', and a very brief description of why the movie matches the user's request."},
]

prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

# Tokenize input
batch_input_tokens = tokenizer(prompt, padding=True, return_tensors="pt").to(model.device)

# Generate response
batch_output_tokens = model.generate(**batch_input_tokens, max_new_tokens=128, do_sample=True)

# Decode response
responses = [
    tokenizer.decode(output_tokens[len(input_tokens) :], skip_special_tokens=True).strip('"')
    for input_tokens, output_tokens in zip(batch_input_tokens["input_ids"], batch_output_tokens)
]

print(responses[0])

In [ ]:
chat